# Recipe 3 — Find Tracks Overlapping a Genomic Region

This notebook walks through querying the FILER coordinate-overlap endpoint to find all
tracks whose intervals intersect a genomic region of interest.

**What you'll learn:**
- How to query the FILER coordinate-overlap endpoint
- How to use `count_only` vs full metadata modes
- How to narrow results by metadata using either Recipe-1-style convenience flags
  or a raw jq `filterString`
- How to explore and summarize overlapping tracks
- How to save results for downstream workflows

**Use when:** you have a specific locus of interest (e.g., a GWAS hit, a candidate enhancer)
and want to discover which FILER tracks have signal there.

**Next steps after this notebook:**
- `01-track-discovery.ipynb` — pre-filter the track universe before querying
- `10-filter-then-overlaps.ipynb` — combine metadata filter + region in one workflow

Run the following before starting:
```
pip install -e ".[dev]"
```

---
## 0. Setup

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

In [ ]:
from filerpy.client import get_overlapping_tracks, ENDPOINTS

In [ ]:
print(f"FILER coordinate-overlap endpoint: {ENDPOINTS['region_overlaps']}")

---
## 1. Hello World — count tracks overlapping a region

The fastest way to verify your connection is to request overlap counts only (`countOnly=1`).
This returns just the track `identifier` and `num_overlaps` — no full metadata fetched.

In [ ]:
results = get_overlapping_tracks(
    "chr1:100000-200000",
    "hg38",
    countOnly=1,
)
print(f"Tracks with overlaps: {len(results)}")
print(results.head(3).to_string())

---
## 2. Configure your query

Edit the parameters below to match your locus of interest.

| Parameter | PHP param | Notes |
|---|---|---|
| `region` | `region` | Format: `chrN:start-end` |
| `genome_build` | `genomeBuild` | `hg19`, `hg38`, or `hg38-lifted` |
| `count_only` | `countOnly` | `1` = counts only (fast), `0` = include download URLs |
| `full_metadata` | `fullMetadata` | `1` = all 36 metadata columns; **required for `filterString`** |

### Metadata filters (two ways)

You can narrow results to specific metadata using either of:

1. **Convenience flags** (`ASSAY`, `CELL_TYPE`, `TISSUE_CATEGORY`, `DATA_SOURCE`, `TRACK_ID`)
   — these mirror Recipe 1's named filters and are joined together with `and` into a single
   jq `filterString` automatically.
2. **Raw `FILTER_STRING`** — a jq boolean expression for full flexibility (e.g. OR logic).
   When set, this overrides the convenience flags.

> ⚠️ Any non-empty filter requires `fullMetadata=1`. Without it, metadata fields are absent
> and every filter evaluates to `null == "value"`, returning empty results. The cell below
> handles this automatically.

In [ ]:
# ── Edit these parameters to match your use case ───────────────────────────
REGION        = "chr1:100000-200000"  # required
GENOME_BUILD  = "hg38"                # required: "hg19", "hg38", or "hg38-lifted"
COUNT_ONLY    = 0                     # 0 = full results, 1 = counts only (fastest)
FULL_METADATA = 1                     # 1 = all metadata columns; required for any filter

# ── Metadata filters: convenience flags (ignored when FILTER_STRING is non-empty) ──
ASSAY           = None  # e.g. "ATAC-seq"
CELL_TYPE       = None  # e.g. "CD14+ monocyte"
TISSUE_CATEGORY = None  # e.g. "Blood"
DATA_SOURCE     = None  # e.g. "ENCODE"
TRACK_ID        = None  # e.g. "NGBLPL2W2SM2WC"

# ── Metadata filters: raw jq filter (overrides convenience flags when non-empty) ──
FILTER_STRING = None  # e.g. '.data_source == "ENCODE" and .assay == "ATAC-seq"'
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
# Build the jq filterString from the parameters above.
# Convenience flags are joined with `and`. FILTER_STRING overrides everything.

JQ_FIELD_MAP = {
    "assay":           ".assay",
    "cell_type":       ".cell_type",
    "tissue_category": ".tissue_category",
    "data_source":     ".data_source",
    "track_id":        ".identifier",
}


def build_filter_string(named: dict) -> str:
    """Combine non-empty named filters into a jq boolean expression joined with `and`."""
    clauses = [f'{JQ_FIELD_MAP[k]} == "{v}"' for k, v in named.items() if v]
    return " and ".join(clauses) if clauses else None


if FILTER_STRING:
    filter_string = FILTER_STRING
    print(f"Using raw FILTER_STRING: {filter_string}")
else:
    filter_string = build_filter_string({
        "assay":           ASSAY,
        "cell_type":       CELL_TYPE,
        "tissue_category": TISSUE_CATEGORY,
        "data_source":     DATA_SOURCE,
        "track_id":        TRACK_ID,
    })
    print(f"Built filter from convenience flags: {filter_string!r}")

# Any non-empty filter requires fullMetadata=1 server-side.
if filter_string:
    if not FULL_METADATA:
        print("filter detected — forcing FULL_METADATA=1")
        FULL_METADATA = 1
    if COUNT_ONLY:
        print("filter detected — forcing COUNT_ONLY=0")
        COUNT_ONLY = 0

In [ ]:
print(f"Region        : {REGION}")
print(f"Genome build  : {GENOME_BUILD}")
print(f"Count only    : {COUNT_ONLY}")
print(f"Full metadata : {FULL_METADATA}")
print(f"Filter string : {filter_string!r}")

---
## 3. Run the query

In [ ]:
print(f"Querying FILER for region: {REGION} ({GENOME_BUILD})...")

In [ ]:
df = get_overlapping_tracks(
    REGION,
    GENOME_BUILD,
    countOnly=COUNT_ONLY,
    fullMetadata=FULL_METADATA,
    **({"filterString": filter_string} if filter_string else {}),
)

In [ ]:
print(f"\nTracks with overlaps: {len(df)}")
print(df.head().to_string())

### 3a. View key columns only

In [ ]:
# Adjust KEY_COLS depending on what was returned
available_cols = df.columns.tolist()

In [ ]:
PREFERRED_COLS = [
    "identifier", "queryRegion", "assay", "cell_type", "tissue_category",
    "life_stage", "data_source", "output_type", "track_name", "num_overlaps",
]
KEY_COLS = [c for c in PREFERRED_COLS if c in available_cols]

In [ ]:
print(f"Displaying columns: {KEY_COLS}")
print(df[KEY_COLS].head(10).to_string())

### 3b. Advanced: raw jq `FILTER_STRING` examples

For power users — set `FILTER_STRING` in section 2 to a raw jq expression for full
boolean flexibility. When `FILTER_STRING` is non-empty it overrides the convenience flags.
Re-run section 2 and 3 after changing it.

```python
# Single condition
FILTER_STRING = '.data_source == "ENCODE"'

# Multiple AND conditions
FILTER_STRING = '.data_source == "ENCODE" and .assay == "ATAC-seq" and .tissue_category == "Blood" and .life_stage == "Adult"'

# OR across data sources (not expressible with convenience flags alone)
FILTER_STRING = '(.data_source == "ENCODE" or .data_source == "Blueprint") and .output_type == "peaks"'

# No filter — return all overlapping tracks
FILTER_STRING = None
```

---
## 4. Explore the results

Before downstream use, understand the shape of your results.
*(Cells below are skipped automatically if metadata columns are absent.)*

In [ ]:
if "data_source" in df.columns:
    print("=== Tracks by data_source ===")
    print(df["data_source"].value_counts().to_string())
else:
    print("data_source not available — re-run with full_metadata=1")

In [ ]:
if "assay" in df.columns:
    print("=== Tracks by assay ===")
    print(df["assay"].value_counts().to_string())
else:
    print("assay not available — re-run with full_metadata=1")

In [ ]:
if "tissue_category" in df.columns:
    print("=== Tracks by tissue_category ===")
    print(df["tissue_category"].value_counts().to_string())
else:
    print("tissue_category not available — re-run with full_metadata=1")

In [ ]:
if "num_overlaps" in df.columns:
    print("=== num_overlaps distribution ===")
    print(df["num_overlaps"].describe())

In [ ]:
# Bar chart — tracks per data source (only when metadata is available)
if "data_source" in df.columns:
    counts = df["data_source"].value_counts()
    fig, ax = plt.subplots(figsize=(8, max(3, len(counts) * 0.4)))
    counts.sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_xlabel("Number of tracks")
    ax.set_title(f"FILER overlapping tracks by data source\n({REGION}, {GENOME_BUILD})")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart — data_source not available")

In [ ]:
# Bar chart — tracks per assay
if "assay" in df.columns:
    assay_counts = df["assay"].value_counts()
    fig, ax = plt.subplots(figsize=(8, max(3, len(assay_counts) * 0.4)))
    assay_counts.sort_values().plot.barh(ax=ax, color="orchid")
    ax.set_xlabel("Number of tracks")
    ax.set_ylabel("Assay")
    ax.set_title(f"FILER overlapping tracks by assay\n({REGION}, {GENOME_BUILD})")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart — assay not available")

---
## 5. Refine your selection

Post-query filtering in pandas — useful for narrowing results without re-querying the API.

In [ ]:
if "life_stage" in df.columns and "num_overlaps" in df.columns:
    # ── Edit these conditions to match what you need ────────────────────────
    df_filtered = df[
        (df["life_stage"] == "Adult") &
        (df["num_overlaps"] >= 1)
    ].copy()
    # ─────────────────────────────────────────────────────────────────────────

    print(f"After refinement: {len(df_filtered)} tracks (from {len(df)} total)")
    print(df_filtered[KEY_COLS].head().to_string())
else:
    df_filtered = df.copy()
    print("Metadata columns not available for refinement — using full result set")

In [ ]:
# Sanity check: are all identifiers unique?
if "identifier" in df_filtered.columns:
    n_dupes = df_filtered["identifier"].duplicated().sum()
    print(f"Duplicate identifiers: {n_dupes}")
    if n_dupes > 0:
        print("Deduplicating...")
        df_filtered = df_filtered.drop_duplicates("identifier")

---
## 6. Save results

### 6a. Save as TSV

In [ ]:
repo_root = Path().resolve().parents[1]
out_dir = repo_root / "output" / "03-coordinate-search"
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
tsv_path = out_dir / "search_results.tsv"
df_filtered.to_csv(tsv_path, sep="\t", index=False)
print(f"Saved {len(df_filtered)} tracks → {tsv_path}")

### 6b. Save query provenance (recommended for reproducibility)

Saves the exact parameters used alongside results so the query is fully reproducible.

In [ ]:
provenance = {
    "query": {
        "region":        REGION,
        "genome_build":  GENOME_BUILD,
        "count_only":    COUNT_ONLY,
        "full_metadata": FULL_METADATA,
        "convenience_flags": {
            "assay":           ASSAY,
            "cell_type":       CELL_TYPE,
            "tissue_category": TISSUE_CATEGORY,
            "data_source":     DATA_SOURCE,
            "track_id":        TRACK_ID,
        },
        "filter_string": filter_string,
    },
    "results": {
        "total_overlapping": len(df),
        "after_refinement":  len(df_filtered),
    },
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "endpoint":  ENDPOINTS["region_overlaps"],
}

In [ ]:
prov_path = out_dir / "search_provenance.json"
with open(prov_path, "w") as f:
    json.dump(provenance, f, indent=2)

In [ ]:
print(f"Provenance saved → {prov_path}")
print(json.dumps(provenance, indent=2))

---
## 7. Batch queries across multiple regions

If you need overlaps for multiple loci, loop and concatenate.
For large BED files, use Recipe 3.2 (batch-query script) instead. 
Please refrain from inputting too many regions as this would overload the server

In [ ]:
regions_of_interest = [
    "chr1:100000-200000",
    "chr2:500000-600000",
    "chr3:1000000-1100000",
]

In [ ]:
frames = []
for region in regions_of_interest:
    chunk = get_overlapping_tracks(region, GENOME_BUILD, countOnly=1)
    frames.append(chunk)
    print(f"  {region} → {len(chunk)} overlapping tracks")

In [ ]:
df_batch = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows across all regions: {len(df_batch)}")
print(df_batch.head().to_string())

---
## 8. Next steps

You now have a `search_results.tsv` and `search_provenance.json` ready for downstream use.

| What to do next                                          | Recipe / Notebook                          |
|----------------------------------------------------------|--------------------------------------------|
| Pre-filter the track universe before querying            | Recipe 1 / `01-track-discovery.ipynb`      |
| Pull the actual intervals from these tracks              | Recipe 2 / `02-track-overlaps.ipynb`       |
| Combine metadata filter + region in one workflow         | Recipe 10 / `10-filter-then-overlaps.ipynb`|
| Install matching tracks locally (giggle + tabix)         | Recipe 11                                  |

```python
# Quick reference: reload your saved results in a future session
import pandas as pd
df = pd.read_csv("output/03-coordinate-search/search_results.tsv", sep="\t")
print(df.head())
```